# MCDI500 — Programación para la ciencia de datos
## Proyecto Transversal — Fase 2: Limpieza y Transformación

**Título del proyecto:** Factores socioeconómicos y de preparación previa asociados al rendimiento académico

| Campo | Detalle |
|---|---|
| **Curso** | MCDI500 — Programación para la ciencia de datos|
| **Docente** | Omar Salinas Silva |
| **Grupo** | Grupo 7 |
| **Integrantes** | Juan de Dios Díaz Ríos · Francisco Fariña Molina · Constanza Moreno Giacometto · Yenne Sepúlveda Jerez |
| **Fecha** | Junio 2026 |
| **Dataset** | Student Performance Dataset (Cortez & Silva, 2008) — UCI Machine Learning Repository |

---
## Índice
1. [Carga del dataset](#1)
2. [Exploración inicial](#2)
3. [Limpieza de datos](#3)
4. [Transformación de variables](#4)
5. [Validación Dataset](#5)
6. [Resumen del pipeline](#6)
7. [Referencias](#7)
8. [Anexo](#8)

<a id='1'></a>
## 1. Carga del dataset

Se cargan los archivos originales desde `data/raw/` utilizando la función `cargar_dataset()`. El separador del CSV es punto y coma (`;`), conforme al formato del dataset UCI.

**Principio fundamental:** los datos crudos en `data/raw/` **nunca se modifican**. Todas las transformaciones se aplican sobre copias de trabajo.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../src"))


from functions import *
from librerias import * #pd para pandas y np para numpy

import importlib
 
importlib.reload(functions)

RUTA_MAT = "../data/raw/student-mat.csv"
RUTA_POR = "../data/raw/student-por.csv"

# Carga de datos
df_mat = cargar_dataset(RUTA_MAT)
df_por = cargar_dataset(RUTA_POR)

print("Dataset cargado exitosamente.")
print(f"  student-mat.csv : {df_mat.shape[0]} filas × {df_mat.shape[1]} columnas")
print(f"  student-por.csv : {df_por.shape[0]} filas × {df_por.shape[1]} columnas")

---
<a id='2'></a>
## 2. Exploración inicial

Antes de cualquier transformación se realiza un diagnóstico completo del dataset. Esta etapa es obligatoria porque permite **tomar decisiones informadas** sobre qué limpiar, qué transformar y qué conservar.

La exploración cubre:
- Estructura general y tipos de datos
- Detección de valores nulos y filas duplicadas
- Validación de rangos según el diccionario de datos
- Detección de outliers con el método IQR
- Análisis de variables categóricas
- Análisis de la variable objetivo G3

In [ ]:
##Llamada a función para mostrar las primeras filas de los datasets
print("Primeras filas del dataset de Matemáticas:")
mostrar_primeras_filas(df_mat,10)




In [ ]:
print("Primeras filas del dataset de Portugués:")
mostrar_primeras_filas(df_por,10)

In [ ]:
# Llamada a funcion para ver cantidad de filas, columnas, duplicados, valores null y tipos de datos
resumen_dataset(df_mat, nombre="Dataset Matemáticas")
resumen_dataset(df_por, nombre="Dataset Portugués")

#### Segun lo anterior no se aplicara ningun tratado de datos en cuanto a valores duplicados ni nulos, ya que ambos dataset no presentan estas incidencias.

In [ ]:
# Estadísticos descriptivos de variables numéricas
print("=== Estadísticos descriptivos — Matemáticas ===")
df_mat.describe().round(2)

In [ ]:
# Estadísticos descriptivos de variables numéricas
print("=== Estadísticos descriptivos — Portugues ===")
df_por.describe().round(2)

In [ ]:

##Llamada a función para validar rangos de datos numericos 

vr_mat = validar_rangos(df_mat, "Matemáticas")
vr_por = validar_rangos(df_por, "Portugués")
#print("\n✓ Todos los valores numéricos están dentro del rango válido.")

In [ ]:
categorical_cols = ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_cols):
    counts = df_mat[col].value_counts()
    counts_df = counts.rename_axis(col).reset_index(name='count')
    palette = sns.color_palette('tab20', len(counts_df))

    plt.subplot(5, 4, i+1)
    sns.barplot(x='count', y=col, data=counts_df, hue=col, dodge=False, palette=palette, legend=False)
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
plt.tight_layout()
os.makedirs('../data/development', exist_ok=True)
plt.savefig('../data/development/distribucion_categoricas.png', dpi=100, bbox_inches='tight')
plt.show()

### A continuación se analizan los outliers

In [ ]:
vars_continuas = ['age', 'absences', 'G1', 'G2', 'G3']
out_mat = detectar_outliers_iqr(df_mat, vars_continuas, "Matemáticas")
out_por = detectar_outliers_iqr(df_por, vars_continuas, "Portugués")

**Interpretación:** La variable `absences` presenta alta asimetría positiva (skewness MAT=3.67, POR=2.02), con valores extremos de hasta 75 ausencias en Matemáticas. Esto justifica un tratamiento específico. El resto de las variables continuas presentan outliers menores dentro de rangos válidos del dominio.

In [ ]:
# Visualización de distribución de absences (outliers)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, df, titulo in zip(axes, [df_mat, df_por], ['Matemáticas', 'Portugués']):
    ax.boxplot(df['absences'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='#AED6F1', color='#2874A6'),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='o', color='orange', alpha=0.6))
    ax.set_title(f'Distribución de Ausencias — {titulo}', fontweight='bold')
    ax.set_ylabel('absences'); ax.set_xticks([])
plt.suptitle('Outliers en variable absences (criterio IQR)', y=1.02)
plt.tight_layout()
os.makedirs('../data/development', exist_ok=True)
plt.savefig('../data/development/fig_outliers_absences.png', dpi=100, bbox_inches='tight')
plt.show()

### A continuación se realizan correlaciones de variables

In [ ]:
matriz_corr_mat = analizar_correlaciones(
    df_mat,
    nombre_dataset="student-mat.csv (Matemáticas)",
    guardar_figura=True,
    ruta_salida="../data/development/correlacion_matematicas.png"
)
#print(matriz_corr_mat)

### Interpretación de las correlaciones con la nota final de Matemáticas(G3)
 
Los resultados muestran que las variables con mayor relación positiva corresponden a las notas obtenidas en períodos anteriores. En particular, la segunda evaluación (**G2**) presenta la correlación más alta con la nota final (**0.90**), seguida por la primera evaluación (**G1**, con 0.80), lo que indica que el desempeño académico previo constituye un fuerte predictor del rendimiento final.
 
En menor medida, se observan correlaciones positivas débiles con el nivel educativo de la madre (**Medu = 0.22**) y del padre (**Fedu = 0.15**), sugiriendo que un mayor nivel educativo de los padres podría estar asociado con mejores resultados académicos.
 
Asimismo, el tiempo dedicado al estudio (**studytime = 0.10**) y la calidad de las relaciones familiares (**famrel = 0.05**) presentan asociaciones positivas, aunque de baja magnitud. Por otra parte, las ausencias (**absences = 0.03**) muestran una relación prácticamente nula con la nota final.
 
Respecto de las variables relacionadas con hábitos y estilo de vida, se observa una correlación negativa débil entre la frecuencia de salidas con amigos (**goout = -0.13**) y las notas finales. De forma similar, el consumo de alcohol durante la semana (**Dalc = -0.05**) y los fines de semana (**Walc = -0.05**) presenta una asociación negativa de baja intensidad con el rendimiento académico.
 
La edad del estudiante (**age = -0.16**) y el tiempo de desplazamiento al establecimiento (**traveltime = -0.12**) también muestran correlaciones negativas débiles.
 
Finalmente, la variable que presenta la asociación negativa más importante con el rendimiento académico es el número de reprobaciones anteriores (**failures = -0.36**), lo que sugiere que los estudiantes con un mayor historial de asignaturas reprobadas tienden a obtener menores calificaciones finales.
 
En general, los resultados indican que el rendimiento académico está principalmente influenciado por el desempeño previo del estudiante y, en menor medida, por factores familiares y antecedentes académicos, mientras que las variables relacionadas con hábitos personales presentan asociaciones más débiles.

In [ ]:
matriz_corr_por = analizar_correlaciones(
    df_por,
    nombre_dataset="student-por.csv (Portugués)", guardar_figura=True,
    ruta_salida="../data/development/correlacion_portugues.png"
)
#print(matriz_corr_por)

### Interpretación de las correlaciones con la nota final de Portugues (G3)
  
Los resultados muestran que las calificaciones obtenidas en evaluaciones anteriores son las variables con mayor asociación positiva con la nota final. En particular, la segunda evaluación (**G2**) presenta una correlación muy fuerte con G3 (**0.92**), seguida por la primera evaluación (**G1 = 0.83**), lo que indica que el desempeño previo del estudiante constituye un importante predictor del rendimiento académico final.
 
Asimismo, el tiempo dedicado al estudio (**studytime = 0.25**) y el nivel educativo de la madre (**Medu = 0.24**) presentan correlaciones positivas débiles a moderadas con la nota final. De manera similar, el nivel educativo del padre (**Fedu = 0.21**) también muestra una asociación positiva, aunque de menor intensidad.
 
Por otra parte, las relaciones familiares (**famrel = 0.06**) presentan una correlación positiva muy débil, mientras que las ausencias (**absences = -0.09**), el estado de salud (**health = -0.09**) y la edad (**age = -0.11**) muestran asociaciones negativas de baja magnitud con el rendimiento académico.
 
Respecto de las variables asociadas a hábitos y tiempo libre, se observa que la frecuencia de salidas con amigos (**goout = -0.09**), el tiempo libre disponible (**freetime = -0.12**) y el tiempo de desplazamiento al establecimiento (**traveltime = -0.12**) presentan correlaciones negativas débiles con la nota final.
 
En cuanto al consumo de alcohol, tanto durante los fines de semana (**Walc = -0.18**) como durante los días laborables (**Dalc = -0.20**) se observa una relación negativa con el rendimiento académico, sugiriendo que mayores niveles de consumo se asocian con menores calificaciones.
 
Finalmente, la variable que presenta la correlación negativa más importante es el número de reprobaciones previas (**failures = -0.39**), lo que indica que los estudiantes con antecedentes de asignaturas reprobadas tienden a obtener peores resultados en la evaluación final.
 
En general, los resultados sugieren que el rendimiento académico en Portugués está principalmente determinado por el desempeño previo del estudiante y por su historial de reprobaciones, mientras que factores relacionados con el entorno familiar y los hábitos personales presentan asociaciones más débiles.

### A continuación se realizara un analisis de la variable objetivo G3

In [ ]:
print("\n=== Análisis G3 = 0 ===")
analizar_g3_cero(df_mat, "MAT")
analizar_g3_cero(df_por, "POR")
print("""
→ Decisión: Se CONSERVAN todos los registros G3=0.
  Son valores válidos (rango 0-20). Eliminarlos introduciría sesgo
  al excluir a los estudiantes con peor desempeño.
  Se crea el flag 'desercion' (G2=0 y G3=0) para análisis diferenciado.
""")

In [ ]:
# Distribución de G3 — histograma comparativo
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, df, titulo, color in zip(axes, [df_mat, df_por],
                                  ['Matemáticas', 'Portugués'], ['#3498DB','#E67E22']):
    ax.hist(df['G3'], bins=21, range=(-0.5, 20.5), color=color, edgecolor='white', alpha=0.85)
    ax.axvline(10, color='red', linestyle='--', linewidth=1.5, label='Umbral aprobación (10)')
    ax.axvline(df['G3'].mean(), color='navy', linestyle=':', linewidth=1.5,
               label=f"Media={df['G3'].mean():.1f}")
    ax.set_title(f'Distribución G3 — {titulo}', fontweight='bold')
    ax.set_xlabel('Calificación final (G3)'); ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=9)
plt.suptitle('Variable Objetivo G3: Calificación Final', fontsize=12)
plt.tight_layout()
os.makedirs('../data/processed', exist_ok=True)
plt.savefig('../data/processed/fig_distribucion_g3.png', dpi=100, bbox_inches='tight')
plt.show()

La distribución de las calificaciones finales en Matemáticas se concentra principalmente entre 8 y 14 puntos, con una media de 10,4. La mayoría de los estudiantes supera el umbral de aprobación establecido en 10 puntos.Las calificaciones finales en Portugués presentan una distribución más concentrada en torno a los 12 puntos, con una media de 11,9. En general, se observa un mayor rendimiento y una mayor proporción de estudiantes sobre el umbral de aprobación de 10 puntos.